# Week 4: HealthConnect Clinic, Initial Data Assessment & Exploration
**AnalystLab Africa Experience Lab, Data Science Track**

This notebook supports the Week 4 Machine Learning Problem Definition Document. It contains the
data loading, quality assessment, and initial single-variable exploration used to identify candidate
target variables and features for the appointment no-show prediction problem.

Per the Week 4 brief, this stage focuses on understanding and initial exploration only, no model
training is performed here.

## 1. Load the Data

In [1]:
import pandas as pd

df = pd.read_csv('HealthConnect_Appointment_Data.csv')
print("Shape:", df.shape)
df.head()

Shape: (5000, 18)


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


## 2. Data Quality Assessment

In [2]:
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())

Data types:
appointment_id               str
patient_id                   str
gender                       str
age                        int64
age_group                    str
appointment_type             str
booking_date                 str
appointment_date             str
appointment_day              str
appointment_time             str
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent                str
reminder_channel             str
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome          str
dtype: object

Missing values:
appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
pre

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())
print("Unique patients:", df['patient_id'].nunique(), "across", len(df), "appointments")

Duplicate rows: 0
Duplicate appointment_id: 0
Unique patients: 1696 across 5000 appointments


**Finding:** No duplicate rows or appointment IDs. 1,696 unique patients across 5,000
appointments, most patients appear more than once in the dataset.

In [4]:
# Confirm reminder_channel missingness is structural, not a data quality issue
print(pd.crosstab(df['reminder_sent'], df['reminder_channel'].isnull()))

reminder_channel  False  True 
reminder_sent                 
No                    0   1366
Yes                3634      0


**Finding:** Every record with `reminder_sent = No` has a null `reminder_channel`, and every
record with `reminder_sent = Yes` has a valid channel. This confirms the 27.3% missingness in
`reminder_channel` is structural (no reminder sent = no channel to record), not a data quality
problem.

In [5]:
# Data integrity check
print("previous_no_shows > previous_appointments violations:",
      (df['previous_no_shows'] > df['previous_appointments']).sum())

previous_no_shows > previous_appointments violations: 0


## 3. Target Variable Exploration

In [6]:
print(df['appointment_outcome'].value_counts())
print()
print(df['appointment_outcome'].value_counts(normalize=True) * 100)

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64


**Finding:** The outcome variable has three categories: No-Show (48.5%), Attended (46.3%), and
Cancelled (5.3%). As discussed in the Problem Definition Document, Cancelled appointments are
proposed to be excluded from modelling, framing this as binary classification (No-Show vs
Attended).

## 4. Candidate Feature Exploration

In [7]:
# booking_lead_days
df['lead_bucket'] = pd.cut(df['booking_lead_days'], bins=[-1,3,7,14,30,1000],
                             labels=['0-3','4-7','8-14','15-30','30+'])
print(df.groupby('lead_bucket', observed=True)['appointment_outcome']
        .apply(lambda x: (x=='No-Show').mean()*100))

lead_bucket
0-3      24.842767
4-7      30.745342
8-14     33.554817
15-30    43.210803
30+      60.494845
Name: appointment_outcome, dtype: float64


**Finding:** Strong signal. No-show rate rises from 24.8% (booked 0-3 days ahead) to 60.5%
(booked 30+ days ahead).

In [8]:
# previous_no_shows
df['prev_noshow_bucket'] = pd.cut(df['previous_no_shows'], bins=[-1,0,1,2,100],
                                    labels=['0','1','2','3+'])
print(df.groupby('prev_noshow_bucket', observed=True)['appointment_outcome']
        .apply(lambda x: (x=='No-Show').mean()*100))

prev_noshow_bucket
0     43.512496
1     53.488372
2     59.360731
3+    68.817204
Name: appointment_outcome, dtype: float64


**Finding:** Strong signal. No-show rate rises from 43.5% (no prior no-shows) to 68.8% (3+
prior no-shows).

In [9]:
# distance_to_clinic_km
df['dist_bucket'] = pd.cut(df['distance_to_clinic_km'], bins=[-1,2,5,10,20,1000],
                             labels=['0-2km','2-5km','5-10km','10-20km','20+km'])
print(df.groupby('dist_bucket', observed=True)['appointment_outcome']
        .apply(lambda x: (x=='No-Show').mean()*100))
print()
print("Bucket sizes:")
print(df['dist_bucket'].value_counts())

dist_bucket
0-2km      51.063830
2-5km      45.276873
5-10km     46.513002
10-20km    49.430797
20+km      57.760814
Name: appointment_outcome, dtype: float64

Bucket sizes:
dist_bucket
5-10km     1692
10-20km    1669
2-5km       921
20+km       393
0-2km       235
Name: count, dtype: int64


**Finding:** Moderate signal. Largely flat (45-49%) until 20+km, where the no-show rate rises
to 57.8% (n=393, a reliable sample size).

In [10]:
# reminder_sent
print(df.groupby('reminder_sent')['appointment_outcome'].apply(lambda x: (x=='No-Show').mean()*100))

reminder_sent
No     51.390922
Yes    47.358283
Name: appointment_outcome, dtype: float64


**Finding:** Weak-to-moderate signal. 51.4% (no reminder) vs 47.4% (reminder sent), a modest
but present difference.

In [11]:
# age_group
print(df.groupby('age_group', observed=True)['appointment_outcome'].apply(lambda x: (x=='No-Show').mean()*100))

age_group
18-24    50.177305
25-34    50.702427
35-44    48.351648
45-54    48.045397
55-64    50.750000
65+      45.124899
Name: appointment_outcome, dtype: float64


**Finding:** Weak signal. Largely flat across age bands (45-51%).

In [12]:
# appointment_type
print(df.groupby('appointment_type')['appointment_outcome'].apply(lambda x: (x=='No-Show').mean()*100))

appointment_type
Diagnostic Test            49.747049
Follow-up                  51.231527
General Consultation       46.644295
Specialist Consultation    47.444444
Name: appointment_outcome, dtype: float64


**Finding:** Weak signal. Largely flat across types (46.6-51.2%).

In [13]:
# waiting_time_minutes - also checked for potential data leakage
print(df.groupby('appointment_outcome')['waiting_time_minutes'].describe())
print()
df['wait_bucket'] = pd.cut(df['waiting_time_minutes'], bins=[-1,15,30,45,60,1000],
                             labels=['0-15','15-30','30-45','45-60','60+'])
print(df.groupby('wait_bucket', observed=True)['appointment_outcome']
        .apply(lambda x: (x=='No-Show').mean()*100))
print()
print("Bucket sizes:")
print(df['wait_bucket'].value_counts())

                      count       mean        std  min   25%   50%   75%   max
appointment_outcome                                                           
Attended             2293.0  24.289141  10.946389  2.0  17.0  24.0  32.0  68.0
Cancelled             261.0  23.214559  10.557896  2.0  16.0  22.0  29.0  55.0
No-Show              2386.0  24.200754  10.814945  2.0  17.0  24.0  31.0  67.0

wait_bucket
0-15     47.064306
15-30    49.190939
30-45    47.777778
45-60    46.212121
60+      66.666667
Name: appointment_outcome, dtype: float64

Bucket sizes:
wait_bucket
15-30    2472
30-45    1260
0-15     1073
45-60     132
60+         3
Name: count, dtype: int64


**Finding:** Inconclusive / potential leakage check. `waiting_time_minutes` is present for all
three outcomes with near-identical means (~24 minutes), suggesting it is an estimated or scheduled
value rather than an actually recorded wait time (a genuine no-show would have no real wait time to
record). This should be confirmed with the project team before use as a model input. The apparent
spike in the 60+ bucket (66.7%) is based on only 3 records and is not a reliable pattern.

## 5. Summary of Findings

| Feature | Signal Strength | Pattern |
|---|---|---|
| booking_lead_days | Strong | 24.8% to 60.5% as lead time increases |
| previous_no_shows | Strong | 43.5% to 68.8% as prior no-shows increase |
| distance_to_clinic_km | Moderate | Flat until 20+km, then 57.8% |
| reminder_sent | Weak-Moderate | 51.4% (No) vs 47.4% (Yes) |
| age_group | Weak | Flat, 45-51% |
| appointment_type | Weak | Flat, 46.6-51.2% |
| waiting_time_minutes | Inconclusive | Mostly flat; needs leakage confirmation |

These findings, along with the data quality assessment above, directly inform the target variable
choice, candidate feature list, and modelling considerations documented in the Week 4 Machine
Learning Problem Definition Document.